In [1]:
print("Multi model")

Multi model


## Multimodel AI - Blood Work Analysis

In [2]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
import base64

load_dotenv()

True

### Encoding the image and send to vision model

In [3]:
with open("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    {"type": "text",      "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
])

response = llm.invoke([message])
print(response.content)

**Patient Information:**

*   **Patient Name:** Rajesh Sharma
*   **Age:** 48
*   **Gender:** Male
*   **Date:** May 7, 2026

**COMPLETE BLOOD COUNT (CBC)**

*   **Hemoglobin:** 15.1 g/dL (Normal: 13.5-17.5) **Within Normal Range**
*   **Hematocrit:** 44% (Normal: 41-53) **Within Normal Range**
*   **WBC:** 6.8 x10^3/uL (Normal: 4.5-11.0) **Within Normal Range**
*   **Platelets:** 220 x10^3/uL (Normal: 150-400) **Within Normal Range**

**LIPID PANEL**

*   **Total Cholesterol:** 238 mg/dL (Normal: <200) **ABOVE NORMAL RANGE**
*   **LDL Cholesterol:** 162 mg/dL (Normal: <100) **ABOVE NORMAL RANGE**
*   **HDL Cholesterol:** 36 mg/dL (Normal: >40) **BELOW NORMAL RANGE**
*   **Triglycerides:** 188 mg/dL (Normal: <150) **ABOVE NORMAL RANGE**

**METABOLIC PANEL**

*   **Glucose (Fasting):** 92 mg/dL (Normal: 70-99) **Within Normal Range**
*   **HbA1c:** 5.3% (Normal: <5.7) **Within Normal Range**
*   **Creatinine:** 1.0 mg/dL (Normal: 0.7-1.3) **Within Normal Range**
*   **eGFR:** 82 mL/min 

### Suggest Diet Plan Agent

- The agent reads the blood work image, categorises the condition, then calls the diet tool.

In [4]:
@tool
def get_diet_recommendation(condition: str) -> dict:
    """Given a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "high_cholesterol": {
            "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
        },
        "high_sugar": {
            "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
        },
        "normal": {
            "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
        },
    }
    return diet_plans.get(condition, diet_plans["normal"])

In [5]:
SYSTEM_PROMPT = """
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan.
"""

diet_agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

In [6]:
result = diet_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text",      "text": "Analyse this blood work report and suggest a diet plan."},
    ])]
})

print(result["messages"][-1].content)

The patient, Rajesh Sharma, has high cholesterol. I recommend a diet plan that includes fruits, vegetables, whole grains, and lean protein. Avoid red meat, fried food, full-fat dairy, and processed snacks.
